# Lab 10: Reasoning Memory

In this lab, you will record agent reasoning traces, search for similar past tasks, and view tool usage statistics. Reasoning memory captures what the agent did and whether it worked — enabling the agent to learn from past executions by reusing successful strategies and avoiding approaches that failed.

## What you will learn

- How to record agent execution traces with `record_agent_trace()`
- How to find similar past traces with `get_similar_traces()`
- How to view tool statistics with `get_tool_stats()`

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

provider_name = os.getenv("LLM_PROVIDER", "openai").lower()
if provider_name == "azure":
    assert os.getenv("AZURE_OPENAI_API_KEY"), "AZURE_OPENAI_API_KEY not set in .env file"
    assert os.getenv("AZURE_OPENAI_ENDPOINT"), "AZURE_OPENAI_ENDPOINT not set in .env file"
else:
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set in .env file"
assert os.getenv("NEO4J_URI"), "NEO4J_URI not set in .env file"
print("Environment loaded successfully")

## Import Dependencies

In [ ]:
from pydantic import SecretStr

from neo4j_agent_memory import MemoryClient, MemorySettings
from neo4j_agent_memory.integrations.microsoft_agent import (
    Neo4jMicrosoftMemory,
    record_agent_trace,
    get_similar_traces,
)

## Configure and Connect

In [ ]:
settings = MemorySettings(
    neo4j={
        "uri": os.environ["NEO4J_URI"],
        "username": os.environ["NEO4J_USERNAME"],
        "password": SecretStr(os.environ["NEO4J_PASSWORD"]),
    },
)

memory_client = MemoryClient(settings)
await memory_client.connect()
print("Connected to Neo4j")

## Create Memory

In [ ]:
memory = Neo4jMicrosoftMemory.from_memory_client(
    memory_client=memory_client,
    session_id="reasoning-demo",
    include_short_term=True,
    include_long_term=True,
    include_reasoning=True,
)

## Record an Agent Trace

Traces let the agent learn from experience — when it encounters a similar task later, it can retrieve successful past approaches and avoid strategies that failed. Use `record_agent_trace()` to capture a completed agent execution. Provide:
- `memory` — the `Neo4jMicrosoftMemory` instance
- `messages` — conversation messages (can be empty for standalone traces)
- `task` — description of what the agent was trying to accomplish
- `tool_calls` — list of tool invocations with name, arguments, result, status, and duration_ms
- `outcome` — the final result
- `success` — whether the task completed successfully

In [ ]:
trace = await record_agent_trace(
    memory=memory,
    messages=[],
    task="Find sci-fi movies about time travel",
    tool_calls=[
        {
            "name": "search_knowledge",
            "arguments": {"query": "time travel sci-fi"},
            "result": ["Interstellar", "The Matrix", "Back to the Future"],
            "status": "success",
            "duration_ms": 150,
        }
    ],
    outcome="Recommended Interstellar based on user preference for Nolan films",
    success=True,
)
print(f"Recorded trace: {trace}")

## Find Similar Traces

Use `get_similar_traces()` to search for reasoning traces from similar past tasks. The search uses the vector embedding of the task description to find semantically similar traces.

In [ ]:
traces = await get_similar_traces(
    memory=memory,
    task="Find action movies with good ratings",
    limit=3,
)

for trace in traces:
    print(f"Task: {trace.task}")
    print(f"Outcome: {trace.outcome}")
    print(f"Success: {trace.success}")
    print(f"Steps: {len(trace.steps)}")

## View Tool Statistics

Reasoning memory tracks aggregate statistics about tool usage. These help identify which tools are reliable and fast vs. which fail frequently or are slow, informing decisions about tool selection and optimization.

In [ ]:
stats = await memory_client.reasoning.get_tool_stats()

for tool in stats:
    print(f"{tool.name}: {tool.success_rate:.0%} success, "
          f"avg {tool.avg_duration_ms}ms")

## Close the Connection

In [ ]:
await memory_client.close()
print("Connection closed")

## Summary

Reasoning memory records agent execution traces including tasks, steps, tool calls, and outcomes. These traces are stored with vector embeddings so the agent can find similar past tasks using semantic search. The agent can learn from past successes and failures, either through automatic context injection or explicit tool calls.